# Part 1 - Multimodal RAG
DATA 266 Lab 2, Pair_Programming_Team_32. The code lives in `part1_rag/src/`; this notebook runs it.
Runs on **Kaggle** (the graded path), Colab, or locally. CPU is enough: Restart & Run All.


In [1]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
except ImportError:
    gpu = "cpu"
print({"commit": commit, "python": sys.version.split()[0], "gpu": gpu, "seed": SEED})

{'commit': '', 'python': '3.13.18', 'gpu': 'cpu', 'seed': 266}


## Setup
- **Kaggle:**
  - Settings: Internet **on**.
  - Add-ons → Secrets: `ANTHROPIC_API_KEY`. `GEMINI_API_KEY` is optional; it's only needed if a figure is missing from the committed transcription cache.
  - Add data:
    - the competition, which supplies the PDFs, `Questions.csv`, `kb_manifest.json` and the validator;
    - a private Kaggle dataset holding `repo.bundle` (made with `git bundle create repo.bundle zoheb`).
- **Colab:** secrets `ANTHROPIC_API_KEY` and `GEMINI_API_KEY`. Upload `repo.bundle` when asked.


In [2]:
import os, glob
BRANCH = "zoheb"                                            # switch to main once merged
ON_KAGGLE, ON_COLAB = os.path.isdir("/kaggle/input"), "google.colab" in sys.modules
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
    try:
        os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
    except Exception:
        pass                                                # only needed on a figure-cache miss
    # the competition files are read-only under /kaggle/input; the code comes from the attached repo.bundle dataset
    os.environ["P1_DATA"] = os.path.dirname(glob.glob("/kaggle/input/**/Questions.csv", recursive=True)[0])
    bundle = glob.glob("/kaggle/input/**/repo.bundle", recursive=True)[0]
    os.chdir("/kaggle/working")
    if not os.path.isdir("data266-lab2-team32"):
        !git clone -q -b {BRANCH} {bundle} data266-lab2-team32
    %cd /kaggle/working/data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai anthropic pydantic
elif ON_COLAB:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files                      # no GitHub token: upload repo.bundle (git bundle create repo.bundle zoheb)
        files.upload()
        !git clone -q -b {BRANCH} repo.bundle data266-lab2-team32
    %cd data266-lab2-team32
    !pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai anthropic pydantic
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import anthropic, chromadb, pymupdf, sentence_transformers, google.genai
from part1_rag.src import common
print({"commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "data": str(common.DATA), "pymupdf": pymupdf.__version__, "chromadb": chromadb.__version__,
       "sentence-transformers": sentence_transformers.__version__, "google-genai": google.genai.__version__, "anthropic": anthropic.__version__,
       "answers": common.CLAUDE_MODEL, "figure transcriptions": common.GEMINI_MODEL})


/kaggle/working/data266-lab2-team32
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-cloud-bigquery 3.46.2 re

## Data
The competition files are not in git. Kaggle reads them from the attached competition data. On Colab, upload the 3 PDFs and `Questions.csv` when asked.


In [3]:
need = [d["file"] for d in common.DOCS.values()] + ["Questions.csv"]
missing = [f for f in need if not (common.DATA / f).exists()]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (common.DATA / name.replace(" (1)", "")).write_bytes(data)
missing = [f for f in need if not (common.DATA / f).exists()]
assert not missing, f"missing data files: {missing}"

## 1.1 Ingest and index
Pages become text chunks; every figure and table is cropped. Figure transcriptions come from the committed
cache `outputs/figure_descriptions.json`, so this step makes no API calls.

In [4]:
from part1_rag.src import ingest, index
chunks, figures = ingest.ingest_all()
index.build_index()

DOC1: 37 pages, 17 figures/tables: Table 1-1, Figure 1-1, Figure 1-2, Figure 1-3, Table 1-2, Figure 1-4, Table 1-3, Figure 1-5, Figure 1-6, Table 1-4, Figure 2-1, Figure 2-2, Figure 2-3, Figure 2-4, Figure 2-5, Figure 2-6, Figure 2-7
DOC2: 7 pages, 10 figures/tables: Table A-1, Figure A-1, Table A-2, Figure A-2, Figure A-3, Figure A-4, Figure A-5, Figure A-6, Figure A-7, Table A-3
DOC3: 7 pages, 9 figures/tables: Table B-1, Figure B-1, Figure B-2, Figure B-3, Figure B-4, Table B-2, Figure B-5, Figure B-5 (corrected), Figure C-1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexed 87 items (36 figures/tables)


Collection(name=knowledge_base)

## 1.2-1.5 Answer Questions.csv
One Claude call per question (40 for the public set). Paid API: roughly $1-2 per full run on Sonnet 5.5.

In [5]:
from part1_rag.src import pipeline
sub, seconds = pipeline.run()
print("median seconds/question:", round(float(np.median(list(seconds.values()))), 2))
print(sub["status"].value_counts().to_dict())
assert pipeline.validate()
sub.head(10)

Q001 answered | NONE | DOC1:p1,DOC1:p2
Q002 answered | NONE | DOC1:p24,DOC1:p3,DOC1:p29
Q003 answered | NONE | DOC1:p20,DOC1:p21,DOC1:p26
Q004 answered | Table 1-1,Table 1-2 | DOC1:Table 1-1,DOC1:p1,DOC1:p2,DOC1:Table 1-2,DOC1:p5
Q005 conflict | Table 1-2,Table A-2,Table B-2 | DOC1:Table 1-2,DOC1:p5,DOC1:p3,DOC1:p24,DOC2:Table A-2,DOC2:p3,DOC3:Table B-2,DOC3:p5
Q006 answered | Table 1-4 | DOC1:Table 1-4,DOC1:p11
Q007 answered | Figure 1-1,Figure A-5 | DOC1:Figure 1-1,DOC1:p2,DOC2:Figure A-5,DOC2:p5
Q008 false_premise | Figure 2-5 | DOC1:Figure 2-5,DOC1:p29,DOC1:p23
Q009 answered | Figure 2-4 | DOC1:Figure 2-4,DOC1:p27,DOC3:p4
Q010 answered | Table 1-3,Table A-2 | DOC1:Table 1-3,DOC1:p7,DOC1:p9,DOC2:Table A-2,DOC2:p3
Q011 answered | Table 1-4 | DOC1:Table 1-4,DOC1:p11
Q012 answered | NONE | DOC1:p1,DOC1:p2
Q013 answered | NONE | DOC1:p1,DOC1:p2,DOC1:p3
Q014 answered | Figure 2-2 | DOC1:p1,DOC1:p21,DOC1:Figure 2-2,DOC1:p36
Q015 answered | Table 1-1 | DOC1:Table 1-1,DOC1:p1
Q016 answered 

,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,"Per DOC1, the crisis began with the US housing...",NONE,"DOC1:p1,DOC1:p2",answered
1,2,C002,Q002,Blanchard gives two reasons economists care ab...,NONE,"DOC1:p24,DOC1:p3,DOC1:p29",answered
2,3,C003,Q003,"Economists use real GDP, which values output a...",NONE,"DOC1:p20,DOC1:p21,DOC1:p26",answered
3,4,C004,Q004,The 2009 hit was severe. World output fell 2.3...,"Table 1-1,Table 1-2","DOC1:Table 1-1,DOC1:p1,DOC1:p2,DOC1:Table 1-2,...",answered
4,5,C005,Q005,"U.S. unemployment rose sharply, then fell back...","Table 1-2,Table A-2,Table B-2","DOC1:Table 1-2,DOC1:p5,DOC1:p3,DOC1:p24,DOC2:T...",conflict
5,6,C006,Q006,"Before the crisis, China's output growth avera...",Table 1-4,"DOC1:Table 1-4,DOC1:p11",answered
6,7,C007,Q007,"Everywhere, not just the U.S. DOC1 says that a...","Figure 1-1,Figure A-5","DOC1:Figure 1-1,DOC1:p2,DOC2:Figure A-5,DOC2:p5",answered
7,8,C008,Q008,Not always. Document 1 describes Okun's law as...,Figure 2-5,"DOC1:Figure 2-5,DOC1:p29,DOC1:p23",false_premise
8,9,C009,Q009,"No. Per DOC1, the CPI and GDP deflator inflati...",Figure 2-4,"DOC1:Figure 2-4,DOC1:p27,DOC3:p4",answered
9,10,C010,Q010,"Before the crisis, Euro area unemployment aver...","Table 1-3,Table A-2","DOC1:Table 1-3,DOC1:p7,DOC1:p9,DOC2:Table A-2,...",answered


In [6]:
import shutil
if ON_KAGGLE:                                               # Kaggle takes the submission from /kaggle/working
    shutil.copy(common.OUTPUTS / "submission.csv", "/kaggle/working/submission.csv")
    print("wrote /kaggle/working/submission.csv")
elif ON_COLAB:
    files.download(str(common.OUTPUTS / "submission.csv"))


wrote /kaggle/working/submission.csv
